**What this notebook does:**
1. Cleans up the datasets for data visualization.


2. Creates a z-score column relative to the company's specific category.
3. Creates a 95% confidence interval for the probable average ratings a company could have.
4. Creates 2 datasets that can be used with these new columns of different sizes.

**Step 1:** clean datasets before merging and feature engineering, so Tableau is displayed without outliers and NULL data.

In [2]:
import pandas as pd
import numpy as np
import gzip
import json

pd.set_option('display.max_columns', None)

def parse(path):
  g = open('Data/' + path, 'r')
  for l in g:
    yield json.loads(l)

def parse_first_n(path, n=10000):
    g = open('Data/' + path, 'r')
    for i, l in enumerate(g):
        if i >= n:
            break
        yield json.loads(l)

In [3]:
texas_metadata = pd.DataFrame(parse("meta-Texas.json"))
texas_reviews = pd.DataFrame(parse_first_n("review-Texas.json", n=1000000)) # is not cleaned yet
#add more or less reviews depending on how well tableu responds to the data.


In [4]:
texas_reviews = texas_reviews.dropna(subset=['rating']) # Drop rows where 'rating' is NaN

In [5]:
texas_metadata = texas_metadata.dropna(subset=['gmap_id', 'name']).drop_duplicates(subset=['gmap_id'], keep='first')



In [6]:
texas_metadata = texas_metadata[texas_metadata['state'] != 'Permanently closed']


In [7]:
texas_metadata = texas_metadata[(texas_metadata['latitude'] >= 25.5) & (texas_metadata['latitude'] <= 36.40)
                            & (texas_metadata['longitude'] >= -106.50) & (texas_metadata['longitude'] <= -93.31)]


In [8]:
texas_metadata.shape[0]

416224

In [9]:

#Extract all unique category labels from california_metadata

category_df = texas_metadata.copy()
category_df = category_df.explode('category')
#Convert everything to lower case and remove blank space 
category_df['category'] = category_df['category'].str.lower().str.strip()

# # # #Remove tiny categories 
category_counts= category_df.get('category').value_counts()


# Setting threshold for the category 
category_counts.describe()
category_counts_threshold = 35 #median, we could alter this if there are too many categories for our deomo


# Map category_count to each category in the df
category_df['category_count'] = category_df['category'].map(category_counts)
category_df = category_df[category_df.get('category_count') > category_counts_threshold]
category_df = category_df.sort_values(by='category_count', ascending=False)




In [10]:

# Calculate mean and standard deviation of each category 
category_stats = category_df.groupby("category")['avg_rating'].agg(['mean', 'std']).reset_index()
category_stats.rename(columns= {'mean': 'category_mean', 'std' : 'category_std'})

,category,category_mean,category_std
0,abrasives supplier,4.304082,0.528346
1,accountant,4.561298,0.690987
2,accounting firm,4.607576,0.699079
3,acupuncture clinic,4.800866,0.281145
4,acupuncturist,4.805882,0.385867
...,...,...,...
1809,yoga instructor,4.610256,0.595065
1810,yoga studio,4.520263,0.609801
1811,youth clothing store,4.420270,0.242125
1812,youth organization,4.343169,0.579546


In [11]:

# Merge back into category data set
mcategory_df = category_df.merge(category_stats, on='category', how='inner') #what happens with a inner join.
mcategory_df = mcategory_df.rename(columns={
    'mean': 'category_mean',
    'std': 'category_std'
})
mcategory_df.shape[0]

952062

In [12]:

# Normalized rating
mcategory_df['rating_z-score'] = (
    (mcategory_df['avg_rating'] - mcategory_df['category_mean']) /
    mcategory_df['category_std']
)
mcategory_df #now this is going to include repeats of companies when it is done

mcategory_df = mcategory_df.groupby('gmap_id').agg({
    'rating_z-score': 'mean'
}).reset_index()

mcategory_df #The reason that the number dropped is becuase explode will create multiple rows and grouping by gmap_id will reduce that back

,gmap_id,rating_z-score
0,0x0:0x3de3e25d19fc7da6,-0.639810
1,0x0:0x48c466d0f3de5f5a,0.514483
2,0x0:0x554de8957fed30a1,0.196436
3,0x0:0x5702eb6d2d1301db,1.148322
4,0x1000000000000001:0xe85d5b0e25befcb7,0.460215
...,...,...
409628,0xaceec0ea6c38bcfd:0xf291bc79aa31fa60,0.692342
409629,0xad33c203e03d89f9:0xb8ee8c778a581ca4,0.707797
409630,0xd0c7e1a240869ad:0xafa498adc8911c71,0.736033
409631,0xdce6e3016b65401:0xf220f369359026f2,1.339574


In [13]:
#We will merge all of this back into texas metadata with an inner merge to add the new z-score and CI columns for the Tableu dataset.

company_stats = (
    texas_reviews #first time using texas reviews, we should definetly do a left merge for CI, since limmited data n=1000000
        .groupby('gmap_id')['rating']
        .agg(['mean', 'std', 'count'])
        .reset_index()
)

def compute_ci(row):
    mean = row['mean']
    std = row['std']
    n = row['count']
    
    if n == 0:
        return None
    
    margin = 1.96 * (std / np.sqrt(n))
    return (mean - margin, mean + margin)

company_stats['CI'] = company_stats.apply(compute_ci, axis=1)

mtexas_metadata = texas_metadata.merge(
    company_stats[['gmap_id', 'CI']],
    on='gmap_id',
    how='inner'
)

# mcategory_df['CI']= mcategory_df['gmap_id'].apply(individual_company_rating_confidence_interval)
# mcategory_df #THIS TAKES TO LONG SPEED IT UP AND THEN MERGE BACK TO TEXAS METADATA




In [ ]:


final_texas_metadata = texas_metadata.merge(mtexas_metadata[['gmap_id', 'CI']], on='gmap_id', how='left').merge(mcategory_df[['gmap_id', 'rating_z-score']], on='gmap_id', how='left')
final_texas_metadata
#shuold be > 409k

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI,rating_z-score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,[Convenience store],4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.070780298646909)",1.587334
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,[Transportation service],4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.141560597293817)",0.702763
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"[Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",-0.716826
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,[Delivery service],2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",-3.818610
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,[Employment agency],2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-2.006615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416219,Texture,"Texture, 2505 W SW Loop 323, Tyler, TX 75701",0x8649ceaecd6cd113:0xcb1ded587cb53b35,None,32.305068,-95.332749,"[Interior designer, Fabric store]",4.9,8,None,"[[Wednesday, 9:30AM–5PM], [Thursday, 9:30AM–5P...",None,NaN,"[0x8649ccfe591fd837:0x650fe45e1c1b0dd7, 0x8649...",https://www.google.com/maps/place//data=!4m2!3...,NaN,0.961050
416220,Sheraton Austin Georgetown Hotel & Conference ...,Sheraton Austin Georgetown Hotel & Conference ...,0x8644d53fe5245059:0x793047042b121b1e,"Refined rooms in an upscale hotel, offering a ...",30.649195,-97.686135,"[Hotel, Indoor lodging, Meeting planning servi...",4.6,948,None,None,None,NaN,None,https://www.google.com/maps/place//data=!4m2!3...,NaN,0.861732
416221,Hana Travel Plaza,"Hana Travel Plaza, 11301 I-40, Amarillo, TX 79118",0x870137dffb31b41b:0xea3a20936f410b50,None,35.193626,-101.706485,"[Truck stop, Truck accessories store]",4.1,808,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x870148f39aeb5bef:0x669fd06662a0d050, 0x8701...",https://www.google.com/maps/place//data=!4m2!3...,NaN,-0.158619
416222,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,None,36.331260,-102.073430,"[Truck stop, Convenience store, Gas station]",3.9,1593,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x8705dff878be024f:0xb42ee8fe9f7898be, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,NaN,0.193181


****BELOW ARE 2 FINAL DATASETS. ONE WITH ONLY THE COMPANIES IN TEXAS REVIEWS WITH NO NAN VALUES this is small_... AND THE OTHER INCLUDES ALL THE DATA IN META BUT NAN VALUES. this is big_.... Choose which one to use BASED ON HOW WELL TABLEAU HANDLES A LOT OF DATA.****

In [27]:
small_test_final_texas_metadata = final_texas_metadata.dropna(subset=['CI'])
small_test_final_texas_metadata = small_test_final_texas_metadata[~small_test_final_texas_metadata['CI'].apply(lambda x: any(pd.isna(i) for i in x))]
small_test_final_texas_metadata['CI'] = small_test_final_texas_metadata['CI'].apply(lambda x: (x[0],5.0) if x[1] > 5.0 else (1.0, x[1]) if x[0] < 1.0 else x)
small_test_final_texas_metadata = small_test_final_texas_metadata.rename(columns={
    'CI': 'CI_for_avg_rating',
    'rating_z-score': 'category_relative_score'
})
small_test_final_texas_metadata


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI_for_avg_rating,category_relative_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,[Convenience store],4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.0)",1.587334
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,[Transportation service],4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.0)",0.702763
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"[Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",-0.716826
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,[Delivery service],2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",-3.818610
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,[Employment agency],2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-2.006615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36880,Larry Pepper A/C & Heating Inc,"Larry Pepper A/C & Heating Inc, 1810 Andrews H...",0x86fbc96cddee47e5:0xe5edb0ba703c33f9,None,31.860806,-102.377652,"[HVAC contractor, Air conditioning repair serv...",3.8,28,None,"[[Friday, 8AM–5PM], [Saturday, Closed], [Sunda...",{'Offerings': ['Repair services']},Open ⋅ Closes 5PM,"[0x86fbc9af1d98a9f5:0xb5e117d567a94ef3, 0x86fb...",https://www.google.com/maps/place//data=!4m2!3...,"(3.1834137103454125, 4.459443432511731)",-1.508179
36881,La Casita Antiques,"La Casita Antiques, 3807 W County Rd 118, Midl...",0x86fbd786f3f4687d:0xf47d2c838c940850,None,31.951504,-102.109351,"[Antique furniture store, Antique store]",3.7,3,None,"[[Friday, Closed], [Saturday, 10:30AM–5:30PM],...","{'Service options': ['In-store shopping'], 'Pl...",Closed ⋅ Opens 10:30AM Sat,"[0x86fbd87949c1eaa5:0xc75b5798311d6628, 0x86fb...",https://www.google.com/maps/place//data=!4m2!3...,"(3.013333333333333, 4.32)",-1.970316
36883,The Conner Team with City First Mortgage Services,The Conner Team with City First Mortgage Servi...,0x86fe712c76442f31:0x83b8a33b9af45707,None,33.490310,-101.936153,[Mortgage lender],5.0,18,None,"[[Friday, 9AM–5:30PM], [Saturday, Closed], [Su...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 5:30PM,"[0x86fe71859c30e2f7:0x98556f8d47fe6027, 0x86fe...",https://www.google.com/maps/place//data=!4m2!3...,"(5.0, 5.0)",0.791927
36884,J & B Plumbing,"J & B Plumbing, 7502 N County Rd 1297, Midland...",0x86fbc36f09935b9b:0x9a2b804b6896fd9e,None,32.040941,-102.286099,[Plumber],5.0,4,None,None,None,Open now,"[0x86fbdb37f6179ec1:0x503944e7edaf5433, 0x86fb...",https://www.google.com/maps/place//data=!4m2!3...,"(5.0, 5.0)",0.895428


In [ ]:
big_test_final_texas_metadata = final_texas_metadata.copy()

def clamp_ci(x): #this fucniton is only needed for big due to NaN handling
    # Skip NaN or non-tuples
    if not isinstance(x, (tuple, list)):
        return x
    
    lower, upper = x
    
    # Clamp values
    lower = max(1.0, lower)
    upper = min(5.0, upper)
    
    return (lower, upper)

big_test_final_texas_metadata['CI'] = (
    big_test_final_texas_metadata['CI'].apply(clamp_ci)
)

# Rename columns
big_test_final_texas_metadata = big_test_final_texas_metadata.rename(columns={
    'CI': 'CI_for_avg_rating',
    'rating_z-score': 'category_relative_score'
})


In [ ]:
print(big_test_final_texas_metadata.shape[0], small_test_final_texas_metadata.shape[0]) #big vs small difference

416224 34376


In [26]:
big_test_final_texas_metadata

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI_for_avg_rating,category_relative_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,[Convenience store],4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.0)",1.587334
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,[Transportation service],4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.0)",0.702763
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"[Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",-0.716826
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,[Delivery service],2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",-3.818610
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,[Employment agency],2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-2.006615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416219,Texture,"Texture, 2505 W SW Loop 323, Tyler, TX 75701",0x8649ceaecd6cd113:0xcb1ded587cb53b35,None,32.305068,-95.332749,"[Interior designer, Fabric store]",4.9,8,None,"[[Wednesday, 9:30AM–5PM], [Thursday, 9:30AM–5P...",None,NaN,"[0x8649ccfe591fd837:0x650fe45e1c1b0dd7, 0x8649...",https://www.google.com/maps/place//data=!4m2!3...,NaN,0.961050
416220,Sheraton Austin Georgetown Hotel & Conference ...,Sheraton Austin Georgetown Hotel & Conference ...,0x8644d53fe5245059:0x793047042b121b1e,"Refined rooms in an upscale hotel, offering a ...",30.649195,-97.686135,"[Hotel, Indoor lodging, Meeting planning servi...",4.6,948,None,None,None,NaN,None,https://www.google.com/maps/place//data=!4m2!3...,NaN,0.861732
416221,Hana Travel Plaza,"Hana Travel Plaza, 11301 I-40, Amarillo, TX 79118",0x870137dffb31b41b:0xea3a20936f410b50,None,35.193626,-101.706485,"[Truck stop, Truck accessories store]",4.1,808,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x870148f39aeb5bef:0x669fd06662a0d050, 0x8701...",https://www.google.com/maps/place//data=!4m2!3...,NaN,-0.158619
416222,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,None,36.331260,-102.073430,"[Truck stop, Convenience store, Gas station]",3.9,1593,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x8705dff878be024f:0xb42ee8fe9f7898be, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,NaN,0.193181


**STOP HERE FOR NOW BELOW IS LATER CODE FOR REVIEW CULTURE BASED ON OUTSIDE DATA SOURCE** (approval pending)

In [ ]:
path = 'Data/simplemaps_uscities_basicv1.93/uscities.csv'

us_cities = pd.read_csv(path)

In [ ]:
us_cities[us_cities['population'] > 500].sort_values(by='population', ascending=False).head()

#Now based on the state we are analyzing we can merge in a sense this data with ours to form a new column.

,city,city_ascii,state_id,state_name,county_fips,county_name,lat,lng,population,density,source,military,incorporated,timezone,ranking,zips,id
0,New York,New York,NY,New York,36081,Queens,40.6943,-73.9249,19268388,10943.7,shape,False,True,America/New_York,1,11229 11228 11226 11225 11224 11222 11221 1122...,1840034016
1,Los Angeles,Los Angeles,CA,California,6037,Los Angeles,34.1141,-118.4068,11984083,3165.7,shape,False,True,America/Los_Angeles,1,91367 90291 90293 90292 91316 91311 90035 9003...,1840020491
2,Chicago,Chicago,IL,Illinois,17031,Cook,41.8375,-87.6866,8609571,4590.3,shape,False,True,America/Chicago,1,60018 60649 60641 60640 60643 60642 60645 6064...,1840000494
3,Miami,Miami,FL,Florida,12086,Miami-Dade,25.7840,-80.2101,6391670,4791.1,shape,False,True,America/New_York,1,33128 33129 33125 33126 33127 33149 33144 3314...,1840015149
4,Houston,Houston,TX,Texas,48201,Harris,29.7860,-95.3885,6227666,1386.2,shape,False,True,America/Chicago,1,77069 77068 77061 77060 77063 77062 77065 7706...,1840020925


What im thinking we do is this: use the function below to create a grid around each business and add that as a coumn to our metadata df. Then from there inlcude the cities from the above df in that column as a list format. **THIS IS WHERE THE ERRORS WILL COME IN**. From the list format we can explode the data, come up with some sort of threshold for what it means to be rural, urban, and a city or discover patterns quickly to categorize "review culture" instead as a way to filter the data for our dashboard.

In [ ]:
#Makes grid over VA. Gemini helped heavily with this
def add_geo_cells(df, lat_col="latitude", lon_col="longitude", cell_size=0.1):
    #get used to using copy more and understanding it
    d = df.copy()
    d["lat_cell"] = (d[lat_col] // cell_size) * cell_size
    d["lon_cell"] = (d[lon_col] // cell_size) * cell_size
    d["geo_cell"] = d["lat_cell"].astype(str) + "_" + d["lon_cell"].astype(str)
    return d

df_meta_cells = add_geo_cells(df_meta)

NameError: name 'df_meta' is not defined

With the code below we could do number of reviews in a geo cell relative to the idetified population, that would showcase how active the people are in reviewing. **MAKE THE PROPORTION** of population density to number_of_reviews in the geocell.

In [ ]:
.agg(
    n_businesses=("gmap_id", "nunique"),
    avg_reviews_per_biz=("num_of_reviews", "mean"),
    median_reviews_per_biz=("num_of_reviews", "median"),
    pct_high_review_biz=("num_of_reviews", lambda x: (x >= 100).mean()),
    avg_rating=("avg_rating", "mean"),

We already have a category column figured out at least for z-score maybe version back to a previous version to include the **PARAMERTER OF REVIEW CULTURE AS A GROUP BY IN THE Z-SCORE CALCULATION**.

In [ ]:
#TODO make a relative surounding area density feature to define how dense the city is relative to its suroudnings
